# Mythic+ Visualizations Overview

This notebook begins the visualization layer of the Mythic+ pipeline by using the Gold fact table to summarize performance for my main characters: **Drayfu**, **Drayth**, and **Draythos**.

## What this notebook does so far

The notebook currently focuses on two main areas:

### 1. Character performance summary
The first section creates a simple summary table for each character using the Gold fact table. This includes:

- total number of runs
- highest key completed
- best Mythic+ score
- average Mythic+ score

This gives a quick high-level view of how each character has performed across all recorded runs.

### 2. Best run by season
The second section compares each character's best run in each season. Since Mythic+ key levels changed after the seasonal scaling update, the notebook creates an **equivalent key** value to make older and newer seasons easier to compare.

For older seasons such as **Battle for Azeroth Season 4** and **Dragonflight Season 1**, the original key level is kept as-is.

For newer seasons after the scaling change, 10 is added to the key level so that the values are more comparable to older key levels.

After that adjustment, the notebook ranks runs by character and season, then keeps the top run for each season. This makes it easier to see each character's strongest performance in each expansion or season.

## Current output tables
At this stage, the notebook produces:

- a **character performance summary**
- a **best run per season** table
- a labeled version of the seasonal best-run table for easier display in charts

## Why this matters
These visualizations are the first step toward building a more complete dashboard. They help answer questions like:

- Which character has done the most runs?
- Which character has completed the highest keys?
- How do my best seasonal runs compare across different expansions?
- Which runs should be highlighted in future dashboards?

## Next steps
Possible next visualizations include:

- best dungeon per character
- average score by season
- timed vs untimed runs
- role-based comparisons
- leaderboard-style charts for my three characters

In [0]:
from pyspark.sql.functions import col, count, max, avg

fact_runs_df = spark.table("workspace.gold_mythic_plus.fact_mythic_runs")

my_characters = ["Drayfu", "Drayth", "Draythos"]

character_performance_df = (
    fact_runs_df
    .filter(col("character_name").isin(my_characters))
    .groupBy("character_name")
    .agg(
        count("*").alias("total_runs"),
        max("keystone_level").alias("highest_key"),
        max("score").alias("best_score"),
        avg("score").alias("avg_score")
    )
    .orderBy(col("highest_key").desc())
)

display(character_performance_df)

In [0]:
from pyspark.sql.functions import col, when, row_number
from pyspark.sql.window import Window

my_characters = ["Drayfu", "Drayth", "Draythos"]

fact_runs_df = spark.table("workspace.gold_mythic_plus.fact_mythic_runs")

fact_runs_adjusted_df = fact_runs_df.withColumn(
    "adjusted_key",
    when(
        col("season").isin("Battle for Azeroth Season 4", "Dragonflight Season 1"),
        col("keystone_level")
    ).otherwise(
        col("keystone_level") + 10
    )
)

window_spec = Window.partitionBy("character_name", "season") \
                    .orderBy(col("adjusted_key").desc())

ranked_runs_df = (
    fact_runs_adjusted_df
    .filter(col("character_name").isin(my_characters))
    .withColumn("rank", row_number().over(window_spec))
)

best_runs_per_season_df = (
    ranked_runs_df
    .filter(col("rank") == 1)
    .select(
        "character_name",
        "season",
        col("adjusted_key").alias("adjusted_highest_key")
    )
)

display(best_runs_per_season_df)

In [0]:
from pyspark.sql.functions import col, when, row_number
from pyspark.sql.window import Window

my_characters = ["Drayfu", "Drayth", "Draythos"]

fact_runs_df = spark.table("workspace.gold_mythic_plus.fact_mythic_runs")

fact_runs_adjusted_df = fact_runs_df.withColumn(
    "adjusted_key",
    when(
        col("season").isin("Battle for Azeroth Season 4", "Dragonflight Season 1"),
        col("keystone_level")
    ).otherwise(
        col("keystone_level") + 10
    )
)

window_spec = Window.partitionBy("character_name", "season") \
                    .orderBy(col("adjusted_key").desc())

ranked_runs_df = (
    fact_runs_adjusted_df
    .filter(col("character_name").isin(my_characters))
    .withColumn("rank", row_number().over(window_spec))
)

best_runs_per_season_df = (
    ranked_runs_df
    .filter(col("rank") == 1)
    .select(
        "character_name",
        "season",
        col("adjusted_key").alias("adjusted_highest_key")
    )
)

display(best_runs_per_season_df)

Databricks visualization. Run in Databricks to view.

In [0]:
from pyspark.sql.functions import col, when, lit, concat, row_number
from pyspark.sql.window import Window

my_characters = ["Drayfu", "Drayth", "Draythos"]

fact_runs_df = spark.table("workspace.gold_mythic_plus.fact_mythic_runs")

fact_runs_adjusted_df = fact_runs_df.withColumn(
    "equivalent_key",
    when(
        col("season").isin("Battle for Azeroth Season 4", "Dragonflight Season 1"),
        col("keystone_level")   # Pre-scaling
    ).otherwise(
        col("keystone_level") + 10   # Post-scaling
    )
)

window_spec = Window.partitionBy("character_name", "season") \
                    .orderBy(col("equivalent_key").desc())

best_runs_df = (
    fact_runs_adjusted_df
    .filter(col("character_name").isin(my_characters))
    .withColumn("rank", row_number().over(window_spec))
    .filter(col("rank") == 1)
    .select(
        "character_name",
        "season",
        col("keystone_level").alias("actual_key"),
        col("equivalent_key")
    )
)

final_df = best_runs_df.withColumn(
    "label_text",
    concat(
        lit("Actual: "),
        col("actual_key"),
        lit(" | Equivalent: "),
        col("equivalent_key")
    )
)

display(final_df)

Databricks visualization. Run in Databricks to view.